In [73]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import HistGradientBoostingRegressor

In [74]:
df = pd.read_csv("/kaggle/input/datasets/mohankrishnathalla/medical-insurance-cost-prediction/medical_insurance.csv")

In [75]:
df

,person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,...,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
0,75722,52,Female,North,Suburban,22700.0,Doctorate,Married,Retired,3,...,0,1,0,1,0,2,0,1,0,0
1,80185,79,Female,North,Urban,12800.0,No HS,Married,Employed,3,...,0,1,1,0,0,1,0,1,1,0
2,19865,68,Male,North,Rural,40700.0,HS,Married,Retired,5,...,0,0,1,1,0,2,1,0,1,0
3,76700,15,Male,North,Suburban,15600.0,Some College,Married,Self-employed,5,...,0,0,0,1,0,0,1,0,0,0
4,92992,53,Male,Central,Suburban,89600.0,Doctorate,Married,Self-employed,2,...,0,1,0,2,0,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,6266,50,Male,West,Urban,127200.0,No HS,Married,Employed,2,...,0,0,0,0,0,1,0,0,0,0
99996,54887,42,Male,East,Suburban,21600.0,HS,Married,Employed,2,...,0,0,0,0,0,0,0,0,0,0
99997,76821,41,Male,West,Rural,81900.0,HS,Divorced,Unemployed,1,...,0,0,0,1,0,1,0,0,0,0
99998,861,51,Female,South,Urban,43400.0,Doctorate,Single,Unemployed,3,...,0,0,1,0,0,2,2,1,0,0


In [76]:
print("information about dataset: ", df.info())
print("shape of dataset: ", df.shape)
print("description: ", df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 54 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   person_id                    100000 non-null  int64  
 1   age                          100000 non-null  int64  
 2   sex                          100000 non-null  object 
 3   region                       100000 non-null  object 
 4   urban_rural                  100000 non-null  object 
 5   income                       100000 non-null  float64
 6   education                    100000 non-null  object 
 7   marital_status               100000 non-null  object 
 8   employment_status            100000 non-null  object 
 9   household_size               100000 non-null  int64  
 10  dependents                   100000 non-null  int64  
 11  bmi                          100000 non-null  float64
 12  smoker                       100000 non-null  object 
 13  

In [77]:
df.isna().sum()

person_id                          0
age                                0
sex                                0
region                             0
urban_rural                        0
income                             0
education                          0
marital_status                     0
employment_status                  0
household_size                     0
dependents                         0
bmi                                0
smoker                             0
alcohol_freq                   30083
visits_last_year                   0
hospitalizations_last_3yrs         0
days_hospitalized_last_3yrs        0
medication_count                   0
systolic_bp                        0
diastolic_bp                       0
ldl                                0
hba1c                              0
plan_type                          0
network_tier                       0
deductible                         0
copay                              0
policy_term_years                  0
p

In [78]:
df['alcohol_freq'].value_counts(dropna=False)

alcohol_freq
Occasional    45078
NaN           30083
Weekly        19833
Daily          5006
Name: count, dtype: int64

In [79]:
df['alcohol_freq'].fillna('None', inplace=True)

/tmp/ipykernel_49/2212994838.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['alcohol_freq'].fillna('None', inplace=True)


In [80]:
df.drop(columns=["person_id", "claims_count", 'avg_claim_amount', 'total_claims_paid', 'annual_medical_cost', 'monthly_premium', 'annual_premium', 'is_high_risk'], inplace=True)

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 46 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   age                          100000 non-null  int64  
 1   sex                          100000 non-null  object 
 2   region                       100000 non-null  object 
 3   urban_rural                  100000 non-null  object 
 4   income                       100000 non-null  float64
 5   education                    100000 non-null  object 
 6   marital_status               100000 non-null  object 
 7   employment_status            100000 non-null  object 
 8   household_size               100000 non-null  int64  
 9   dependents                   100000 non-null  int64  
 10  bmi                          100000 non-null  float64
 11  smoker                       100000 non-null  object 
 12  alcohol_freq                 100000 non-null  object 
 13  

In [82]:
# df['annual_medical_cost'].skew()

In [83]:
# sns.histplot(df['annual_medical_cost'], bins=50, kde=True)
# plt.title("Distribution of Annual Medical Cost")
# plt.show()

In [84]:
# df['annual_medical_cost'].hist(bins=50)

In [85]:
# sns.boxplot(df['annual_medical_cost'])
# plt.title("box plot of annual medical")
# plt.show()

In [86]:
# annual_test = np.log1p(df['annual_medical_cost'])
# annual_test.skew()
# sns.histplot(annual_test, bins=50, kde=True)
# plt.title("Skewness of annual medical cost after applying log function")
# plt.show()

In [87]:
# corr_matrix = df.corr(numeric_only=True)
# top_corr = corr_matrix['annual_medical_cost'].sort_values(ascending=False)
# print(top_corr.head(15))

In [88]:
# plt.figure(figsize=(8,5))
# sns.barplot(x=top_corr[1:11].values, y=top_corr[1:11].index)
# plt.title("Top 10 Features Correlated with Annual Medical Cost")
# plt.show()

In [89]:
cols = df.select_dtypes(include='object').columns
for col in cols:
    print(df[col].unique())

['Female' 'Male' 'Other']
['North' 'Central' 'West' 'South' 'East']
['Suburban' 'Urban' 'Rural']
['Doctorate' 'No HS' 'HS' 'Some College' 'Masters' 'Bachelors']
['Married' 'Single' 'Divorced' 'Widowed']
['Retired' 'Employed' 'Self-employed' 'Unemployed']
['Never' 'Current' 'Former']
['None' 'Weekly' 'Daily' 'Occasional']
['PPO' 'POS' 'HMO' 'EPO']
['Bronze' 'Gold' 'Platinum' 'Silver']


In [90]:
cols

Index(['sex', 'region', 'urban_rural', 'education', 'marital_status',
       'employment_status', 'smoker', 'alcohol_freq', 'plan_type',
       'network_tier'],
      dtype='object')

In [91]:
df['education'] = df['education'].map({'No HS': 0, 'HS': 1, 'Some College': 2, 'Bachelors': 3, 'Masters': 4, 'Doctorate': 5})

In [92]:
df['alcohol_freq'] = df['alcohol_freq'].map({'None': 0, 'Occasional': 1, 'Weekly': 2, 'Daily': 3})
df['network_tier'] = df['network_tier'].map({'Bronze': 0, 'Silver': 1, 'Gold': 2, 'Platinum': 3})
df['smoker'] = df['smoker'].map({'Never': 0, 'Former': 1, 'Current': 2})

In [93]:
encoded_data = pd.get_dummies(df, columns=cols, drop_first=True, dtype=int)

In [115]:
encoded_data.isna().sum().sum()

np.int64(0)

In [95]:
X = encoded_data.drop(columns=['risk_score'])
y = encoded_data['risk_score']

In [96]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [97]:
scaler = StandardScaler()

In [98]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [99]:
lr = LinearRegression()

In [100]:
lr.fit(X_train_scaled, y_train)

LinearRegression()

In [101]:
y_pred = lr.predict(X_test_scaled)

In [102]:
mean_absolute_error(y_test, y_pred)

0.03075162278012315

In [103]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(0.04492080110461456)

In [104]:
r2_score(y_test, y_pred)

0.9679428771312765

In [105]:
y_train_log = np.log1p(y_train)

In [106]:
lr1 = LinearRegression()

In [107]:
lr1.fit(X_train_scaled, y_train_log)

LinearRegression()

In [108]:
y_pred_log = lr1.predict(X_test_scaled)

In [109]:
y_pred_actual = np.expm1(y_pred_log)

In [110]:
mean_absolute_error(y_test, y_pred_actual)

0.03834604928516328

In [111]:
np.sqrt(mean_squared_error(y_test, y_pred_actual))

np.float64(0.06464934703522185)

In [112]:
r2_score(y_test, y_pred_actual)

0.9336015651992235

In [116]:
rfr = RandomForestRegressor(max_depth=15, random_state=42, n_estimators=100)

In [117]:
rfr.fit(X_train, y_train)

RandomForestRegressor(max_depth=15, random_state=42)

In [118]:
y_pred_rfr = rfr.predict(X_test)

In [119]:
mean_absolute_error(y_test, y_pred_rfr)

7.665171666741229e-05

In [120]:
np.sqrt(mean_squared_error(y_test, y_pred_rfr))

np.float64(0.001420554014967004)

In [121]:
r2_score(y_test, y_pred_rfr)

0.9999679413625465

In [122]:
regr = HistGradientBoostingRegressor()

In [123]:
regr.fit(X_train, y_train)

HistGradientBoostingRegressor()

In [124]:
y_pred1 = regr.predict(X_test)

In [125]:
r2_score(y_test, y_pred1)

0.9998708331268599

In [126]:
y_train_pred = regr.predict(X_train)
r2_score(y_train, y_train_pred)

0.9998815451961994

In [128]:
X_train.head()

,age,income,household_size,dependents,bmi,visits_last_year,hospitalizations_last_3yrs,days_hospitalized_last_3yrs,medication_count,systolic_bp,...,smoker_2,alcohol_freq_1,alcohol_freq_2,alcohol_freq_3,plan_type_HMO,plan_type_POS,plan_type_PPO,network_tier_1,network_tier_2,network_tier_3
75220,73,67700.0,2,0,29.3,1,0,0,0,135.0,...,0,1,0,0,0,0,0,0,0,1
48955,43,90200.0,2,0,24.1,1,0,0,0,120.0,...,0,0,0,0,0,0,1,0,0,0
44966,70,64200.0,2,0,36.9,2,0,0,2,116.0,...,0,0,0,0,0,0,1,1,0,0
13568,32,26000.0,2,0,35.5,1,0,0,0,114.0,...,0,1,0,0,1,0,0,0,1,0
92727,33,32100.0,3,1,32.5,0,0,0,1,113.0,...,0,1,0,0,0,1,0,0,0,0


In [129]:
import joblib

base_features = ['age', 'bmi', 'systolic_bp', 'ldl', 'hba1c', 'chronic_count']
encoded_cols = [col for col in X_train.columns if col.startswith(('smoker_', 'alcohol_freq_'))]
features = base_features + encoded_cols

print("Final Features:", features)

deploy_model = HistGradientBoostingRegressor(random_state=42)
deploy_model.fit(X_train[features], y_train)
print("Deploy Model R2:", r2_score(y_test, deploy_model.predict(X_test[features])))

joblib.dump({'model': deploy_model, 'features': features}, 'risk_model.joblib')
print("Model saved successfully!")

Final Features: ['age', 'bmi', 'systolic_bp', 'ldl', 'hba1c', 'chronic_count', 'smoker_1', 'smoker_2', 'alcohol_freq_1', 'alcohol_freq_2', 'alcohol_freq_3']
Deploy Model R2: 0.9998674768024152
Model saved successfully!
